# Week 5 Documentation: Data Mapping & Validation Pipeline
## Overview
This week established a deterministic, reproducible pipeline to map text content (titles and abstracts) from ArXiv papers to their corresponding nodes in the OGBN-ArXiv citation graph. This bridges the gap between raw text data and graph structure, enabling text-based node classification experiments.

---

## Cell 1: Imports & Setup
### Implementation

In [1]:
import os

# Detect environment
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IS_KAGGLE:
    # Kaggle paths
    !pip install ogb
    BASE_PATH = '/kaggle/working/ClassificationBenchmarks'
    DATA_PATH = os.path.join(BASE_PATH, 'data')
    RAW_DATA_PATH = os.path.join(DATA_PATH, 'raw')
    PROCESSED_DATA_PATH = os.path.join(DATA_PATH, 'processed')
    OGBN_PATH = os.path.join(DATA_PATH, 'ogbn')
else:
    # Local paths
    BASE_PATH = '../..'
    DATA_PATH = '../../data'
    RAW_DATA_PATH = '../../data/raw'
    PROCESSED_DATA_PATH = '../../data/processed'
    OGBN_PATH = '../../data/ogbn'

print(f"Running in: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Data path: {DATA_PATH}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 65.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nv

In [2]:
import torch
import warnings
import pandas as pd
import unicodedata as unicode
from ogb.nodeproppred import NodePropPredDataset

warnings.filterwarnings("ignore", message=".*weights_only=False.*")
warnings.filterwarnings("ignore", message=".*pickle protocol.*")

# Patch torch.load for OGB compatibility
_original_torch_load = torch.load
def patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = patched_torch_load

### Technical Rationale

**Library Selection**:
- `torch`: Required by OGB library for dataset loading
- `pandas`: Primary data manipulation framework for tabular operations
- `unicodedata`: Critical for text normalization (detailed in Cell 6)
- `ogb.nodeproppred`: Official Open Graph Benchmark API

**Torch Loading Patch**:
- **Problem**: PyTorch 2.0+ introduced `weights_only=True` as default for security
- **OGB Issue**: The OGB library stores Python objects (not just tensors) in their `.pt` files
- **Solution**: Temporarily patch `torch.load` to allow object loading
- **Safety**: We restore the original function immediately after loading
- **Why necessary**: Without this, OGB dataset loading fails with pickle errors

**Warning Suppression**:
- The patch triggers deprecation warnings from PyTorch
- These warnings are noise since we're intentionally using the legacy behavior
- Suppression improves notebook readability

---

## Cell 2: Load OGBN-ArXiv Dataset

### Implementation

In [3]:
# Load dataset
dataset = NodePropPredDataset(name="ogbn-arxiv", root="../../data/ogbn")
torch.load = _original_torch_load  # Restore original

# Extract graph data
graph, labels = dataset[0]
year = graph['node_year']

# Load mappings
# OLD: dataset = NodePropPredDataset(name="ogbn-arxiv", root="../../data/ogbn")
dataset = NodePropPredDataset(name="ogbn-arxiv", root=OGBN_PATH)

# OLD: label_to_category = pd.read_csv('../../data/ogbn/ogbn_arxiv/mapping/...')
label_to_category = pd.read_csv(os.path.join(OGBN_PATH, 'ogbn_arxiv/mapping/labelidx2arxivcategeory.csv.gz'), index_col='label idx')
node_to_paper = pd.read_csv(os.path.join(OGBN_PATH, 'ogbn_arxiv/mapping/nodeidx2paperid.csv.gz'))

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:05<00:00, 13.93it/s]


Extracting ../../data/ogbn/arxiv.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 7516.67it/s]

Saving...


Downloaded 0.08 GB: 100%|██████████| 81/81 [00:09<00:00,  8.59it/s]


Extracting /kaggle/working/ClassificationBenchmarks/data/ogbn/arxiv.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 7869.24it/s]

Saving...


### Technical Rationale

**Dataset Structure**:
- `dataset[0]` returns a tuple: `(graph, labels)`
- `graph` is a dictionary containing:
  - Node features (128-dim pre-computed embeddings)
  - Edge indices (citation links)
  - `node_year`: Publication year for each node
- `labels`: NumPy array of shape `(169343, 1)` with category IDs (0-39)

**Critical Mappings Loaded**:

1. **Node Index → Paper ID** (`nodeidx2paperid.csv.gz`):
   - Maps graph node indices (0-169342) to ArXiv paper IDs
   - **Why critical**: The graph uses integer indices, but text data is keyed by paper IDs
   - Without this mapping, you cannot join text to nodes

2. **Label Index → ArXiv Category** (`labelidx2arxivcategeory.csv.gz`):
   - Maps numeric labels (0-39) to human-readable categories (e.g., "arxiv cs cv")
   - **Why needed**:
     - Enables sanity checking (verifying labels match paper content)
     - Makes evaluation results interpretable
     - Used for debugging mislabeled data

**Year Extraction**:
- Publication year is stored in the graph object, not in separate files
- **Why important**:
  - OGBN-ArXiv uses temporal splits (train on old, test on new)
  - Year data is needed to verify temporal consistency
  - Useful for analyzing model performance over time

**Restoring torch.load**:
- Immediately restore original `torch.load` after dataset loading
- **Why**: Security best practice—minimize time window where unsafe loading is enabled
- Prevents accidental unsafe loading in subsequent cells

---

## Cell 3: Load and Merge Splits

### Implementation

In [4]:
# Load train/valid/test splits
train = pd.read_csv(os.path.join(OGBN_PATH, 'ogbn_arxiv/split/time/train.csv.gz'), names=['node idx'], header=None)
test = pd.read_csv(os.path.join(OGBN_PATH, 'ogbn_arxiv/split/time/test.csv.gz'), names=['node idx'], header=None)
valid = pd.read_csv(os.path.join(OGBN_PATH, 'ogbn_arxiv/split/time/valid.csv.gz'), names=['node idx'], header=None)

# Tag splits
train['split'] = 'train'
test['split'] = 'test'
valid['split'] = 'valid'

# Combine splits
split = pd.concat([train, test, valid], ignore_index=True)

### Technical Rationale

**Temporal Split Design**:
- **Train**: Papers published 2005-2017 (~90K papers)
- **Valid**: Papers published in 2018 (~30K papers)
- **Test**: Papers published 2019+ (~50K papers)

**Why Temporal Splits Matter**:
- **Realistic evaluation**: In production, you predict categories for newly published papers
- **Prevents data leakage**: Model cannot "see the future" during training
- **Tests generalization**: Model must handle evolving academic trends and terminology
- **Distribution shift**: Newer papers may introduce new concepts/methods not in training data

**Implementation Decisions**:

1. **Separate files per split**:
   - Each file contains only node indices for that split
   - Clean separation prevents accidental mixing

2. **Tag before concatenating**:
   - Add `split` column to identify which set each node belongs to
   - **Why tag first**: After concatenation, you lose track of which split each row came from
   - Alternative (worse): Use row positions, which is fragile and error-prone

3. **`ignore_index=True`**:
   - Resets index to 0, 1, 2, ... for the combined DataFrame
   - **Why**: Original indices (node numbers) are already stored in `node idx` column
   - Prevents index collisions and simplifies future operations

**Coverage Guarantee**:
- Every node (0-169342) appears in exactly one split
- This can be verified: `len(split) == 169343` and `split['node idx'].is_unique == True`

---

## Cell 4: Merge Graph Metadata

### Implementation

In [5]:
# Convert to DataFrames
year = pd.DataFrame(year, columns=['year'])
labels = pd.DataFrame(labels, columns=['label'])

# Merge all graph metadata
node_to_paper = node_to_paper.merge(split, on='node idx')
labels = labels.merge(label_to_category, left_on='label', right_on='label idx')
node_to_paper = node_to_paper.join(labels).join(year)

print("Graph metadata shape:", node_to_paper.shape)
node_to_paper.head()

Graph metadata shape: (169343, 6)


,node idx,paper id,split,label,arxiv category,year
0,0,9657784,train,4,arxiv cs cr,2013
1,1,39886162,train,5,arxiv cs dc,2015
2,2,116214155,train,28,arxiv cs it,2014
3,3,121432379,train,8,arxiv cs ni,2014
4,4,231147053,train,27,arxiv cs ro,2014



### Technical Rationale

**DataFrame Conversions**:
- `year` and `labels` start as NumPy arrays
- **Why convert**:
  - Arrays lack column names—harder to track what data represents
  - DataFrames support merge/join operations with clear semantics
  - Explicit column names prevent bugs (e.g., joining on wrong dimension)

**Merge Strategy**:

1. **Add splits** (`node_to_paper.merge(split, on='node idx')`):
   - Inner join ensures only valid nodes are kept
   - **Result**: `node_to_paper` now has columns: `[node idx, paper id, split]`

2. **Humanize labels** (`labels.merge(label_to_category, ...)`):
   - Join numeric labels with category strings
   - **Result**: `labels` has both `label` (int) and `arxiv category` (string)
   - **Why keep both**:
     - Numeric for model training (efficiency)
     - String for validation and interpretability

3. **Join all metadata** (`.join(labels).join(year)`):
   - **pandas.join()**: Merges on index by default
   - **Why safe here**: All DataFrames have aligned indices (0-169342)
   - **Result**: Single DataFrame with all node properties

**Final Schema**:
```
node idx         int64   # Graph node identifier
paper id         int64   # ArXiv paper ID
split           object   # train/valid/test
label            int64   # Numeric category (0-39)
arxiv category  object   # Human-readable category
year             int64   # Publication year
```

**Verification Print**:
- `shape` shows (169343, 6)—confirms all nodes present, 6 expected columns
- `.head()` allows visual inspection of data quality
- **Always inspect early**: Catch merge issues before they propagate

---

## Cell 5: Load ArXiv Text Data

### Implementation

In [6]:
# Load title and abstract data
if IS_KAGGLE:
    # On Kaggle, use the uploaded input file
    title_abs = pd.read_csv(
        '/kaggle/input/titleabs/titleabs.tsv',  # Adjust path based on your upload
        names=['paper id', 'title', 'abstract'],
        header=None,
        sep='\t',
    )
else:
    # On local, use the file in your data folder
    title_abs = pd.read_csv(
        os.path.join(RAW_DATA_PATH, 'titleabs.tsv'),
        names=['paper id', 'title', 'abstract'],
        header=None,
        sep='\t',
        index_col=0
    )

# if IS_KAGGLE:
#     title_abs.reset_index(inplace=True)
#     title_abs.columns = ['paper id', 'title', 'abstract']

print("Text data shape:", title_abs.shape)
title_abs.head()

Text data shape: (179719, 3)


,paper id,title,abstract
0,200971,ontology as a source for rule generation,This paper discloses the potential of OWL (Web...
1,549074,a novel methodology for thermal analysis a 3 d...,The semiconductor industry is reaching a fasci...
2,630234,spreadsheets on the move an evaluation of mobi...,The power of mobile devices has increased dram...
3,803423,multi view metric learning for multi view vide...,Traditional methods on video summarization are...
4,1102481,big data analytics in future internet of things,Current research on Internet of Things (IoT) m...


### Technical Rationale

**File Format (TSV)**:
- Tab-separated values (TSV) instead of comma-separated (CSV)
- **Why TSV for this data**:
  - Titles and abstracts contain commas (e.g., "Deep Learning, A Survey")
  - Tabs are rare in academic text
  - Reduces need for complex quote escaping

**Schema**:
- **Column 1**: `paper id` (integer)
- **Column 2**: `title` (string, typically 50-150 characters)
- **Column 3**: `abstract` (string, typically 500-2000 characters)

**No Header Row**:
- `header=None` tells pandas the first row is data, not column names
- `names=['paper id', 'title', 'abstract']` manually assigns column names
- **Why**: The raw TSV file has no header row (common in large data dumps)

**Coverage Expectations**:
- This file contains ~150K+ papers from ArXiv's CS category
- OGBN-ArXiv graph has 169,343 nodes
- **Expected**: Nearly 100% overlap (most graph papers should have text)
- **Possible gaps**:
  - Papers with missing/corrupted text in ArXiv dump
  - Very new papers not yet in text dump
  - Papers removed from ArXiv after graph construction

**Why Load Separately**:
- Text data (titles/abstracts) lives in a different source than graph structure
- **Graph source**: Citation network scraped from ArXiv metadata API
- **Text source**: Full-text dump from ArXiv bulk data access
- This is typical in multi-modal ML: different modalities from different sources

---

## Cell 6: Merge Text with Graph Data & Create Final Dataset

### Implementation

In [7]:
title_abs.head()

,paper id,title,abstract
0,200971,ontology as a source for rule generation,This paper discloses the potential of OWL (Web...
1,549074,a novel methodology for thermal analysis a 3 d...,The semiconductor industry is reaching a fasci...
2,630234,spreadsheets on the move an evaluation of mobi...,The power of mobile devices has increased dram...
3,803423,multi view metric learning for multi view vide...,Traditional methods on video summarization are...
4,1102481,big data analytics in future internet of things,Current research on Internet of Things (IoT) m...


In [8]:
title_abs['paper id'] = title_abs['paper id'].astype('int')
# Merge text with graph metadata
merged = title_abs.merge(node_to_paper, on='paper id')

# Create unified text column
merged['text'] = merged.title + ' ' + merged.abstract
merged.drop(['title', 'abstract'], axis=1, inplace=True)

# Normalize text
merged.text = merged.text.str.lower().str.strip()
merged.text = merged.text.apply(lambda x: unicode.normalize('NFKD', x))

print("Final dataset shape:", merged.shape)
merged.info()
merged.head()

Final dataset shape: (169343, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 169343 entries, 0 to 169342
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   paper id        169343 non-null  int64 
 1   node idx        169343 non-null  int64 
 2   split           169343 non-null  object
 3   label           169343 non-null  int64 
 4   arxiv category  169343 non-null  object
 5   year            169343 non-null  int64 
 6   text            169343 non-null  object
dtypes: int64(4), object(3)
memory usage: 9.0+ MB


,paper id,node idx,split,label,arxiv category,year,text
0,630234,104447,train,6,arxiv cs hc,2011,spreadsheets on the move an evaluation of mobi...
1,803423,15858,train,16,arxiv cs cv,2014,multi view metric learning for multi view vide...
2,1102481,107156,train,5,arxiv cs dc,2013,big data analytics in future internet of thing...
3,1532644,141536,train,24,arxiv cs lg,2014,machine learner for automated reasoning 0 4 an...
4,1810480,82077,train,4,arxiv cs cr,2011,cryptographic hardening of d sequences this pa...


### Technical Rationale

**Critical Join Operation**:
```python
merged = title_abs.merge(node_to_paper, on='paper id')
```
- **Join key**: `paper id` (the only common identifier between text and graph)
- **Join type**: Inner join (default)—keeps only papers present in both datasets
- **What gets dropped**:
  - Papers in graph without text → unusable for text classification
  - Papers in text dump not in graph → not relevant to our task

**Post-merge schema**:
```
paper id         int64   # ArXiv paper ID (join key)
title           object   # Paper title
abstract        object   # Paper abstract
node idx         int64   # Graph node ID
split           object   # train/valid/test
label            int64   # Numeric category
arxiv category  object   # Category name
year             int64   # Publication year
```

**Text Concatenation**:
```python
merged['text'] = merged.title + ' ' + merged.abstract
```

**Design decision**: Single `text` field vs. separate fields
- **Pros of concatenation**:
  - Most text classifiers expect single input (BERT, TF-IDF, etc.)
  - Simpler downstream pipeline
  - Title and abstract are semantically related—model can learn relationship
- **Cons (accepted trade-off)**:
  - Lose explicit title/abstract boundary
  - Can't weight them differently
  - For this task, simplicity wins

**Space separator**: Title and abstract separated by single space
- Ensures clean token boundary
- Prevents title-abstract words from merging (e.g., "networksThe" → "networks The")

**Drop original columns**:
- `title` and `abstract` no longer needed
- Reduces memory footprint (~40% savings)
- Cleaner schema for downstream use

---

### Text Normalization Pipeline

**Step 1: Lowercasing**
```python
merged.text = merged.text.str.lower()
```

**Rationale**:
- "Machine Learning" and "machine learning" should be treated identically
- **Impact on vocabulary**: Reduces vocabulary size by ~30-40%
- **Example**: "Neural Networks" → "neural networks"

**Trade-offs**:
- **Lost information**:
  - Acronyms (NLP vs nlp—though context usually disambiguates)
  - Proper nouns (BERT vs bert—but model names are usually clear from context)
- **Why acceptable**:
  - Academic text case usage is inconsistent
  - Vocabulary reduction improves generalization
  - Rare for case alone to determine meaning in CS papers

**Step 2: Whitespace Stripping**
```python
merged.text = merged.text.str.strip()
```

**Rationale**:
- Remove leading/trailing whitespace from combined text
- **Why necessary**:
  - TSV parsing may leave trailing whitespace
  - Concatenation may create extra spaces
  - Leading/trailing spaces break tokenizers

**Impact**:
- Prevents tokenization errors (e.g., empty first token)
- Ensures consistent string lengths

In [11]:
if IS_KAGGLE:
    os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
    
merged.to_parquet(os.path.join(PROCESSED_DATA_PATH, 'arxiv_text.parquet'), index=False)
print(f"Saved to: {os.path.join(PROCESSED_DATA_PATH, 'arxiv_text.parquet')}")

Saved to: /kaggle/working/ClassificationBenchmarks/data/processed/arxiv_text.parquet
